In [1]:
%pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip show langchain-text-splitters

Name: langchain-text-splitters
Version: 1.1.2
Summary: LangChain text splitting utilities
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\HP\OneDrive\Desktop\LLM_Project\venv\Lib\site-packages
Requires: langchain-core
Required-by: langchain-classic


In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

C:\Users\HP\AppData\Local\Temp\ipykernel_2348\1498505327.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\HP\OneDrive\Desktop\LLM_Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
loader = PyPDFLoader(r"C:\Users\HP\OneDrive\Desktop\LLM_Project\documents\doc.pdf")
documents = loader.load()
print(f"✅ Loaded {len(documents)} pages")

✅ Loaded 45 pages


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=20
)
docs = text_splitter.split_documents(documents)
print(f"✅ Created {len(docs)} chunks")

✅ Created 196 chunks


In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embeddings model loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6595.22it/s]


✅ Embeddings model loaded


In [ ]:
os.environ["PINECONE_API_KEY"] = "YOUR_PINECONE_KEY_HERE"  

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "langchainvector"

# Create index if it doesn't exist
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)
print("✅ Pinecone connected!")

✅ Pinecone connected!


In [ ]:
vectorstore = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name=index_name
)
print(" SUCCESS! Your PDF data is uploaded to Pinecone!")

🎉 SUCCESS! Your PDF data is uploaded to Pinecone!


In [9]:
query = "what is this document about?"  # 👈 change to something relevant
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
the use of multiple cylinders, in which the output pins of  one cylinder are connected to the input 
pins of the next. Figure 2.7 shows a three -cylinder system. The left half of the figure shows a 
position in which the input from the operator to the first pin (plaintext letter a) is routed through 
the three cylinders to appear at the output of the second pin (ciphertext letter B). 
With multiple cylinders, the one closest to the operator input rotates one pin position with each

--- Result 2 ---
the use of multiple cylinders, in which the output pins of  one cylinder are connected to the input 
pins of the next. Figure 2.7 shows a three -cylinder system. The left half of the figure shows a 
position in which the input from the operator to the first pin (plaintext letter a) is routed through 
the three cylinders to appear at the output of the second pin (ciphertext letter B). 
With multiple cylinders, the one closest to the operator input rotates one pin position wi

In [10]:
## Cosine Similarity Retrive Results from vectorDB

def retrieve_query(query,k=2):
    matching_results=index.similarity_search(query,k=k)
    return matching_results

In [ ]:


query = "What is this document about?"

# Step 1 - find relevant chunks
docs = vectorstore.similarity_search(query, k=3)

# Step 2 - print results
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
the use of multiple cylinders, in which the output pins of  one cylinder are connected to the input 
pins of the next. Figure 2.7 shows a three -cylinder system. The left half of the figure shows a 
position in which the input from the operator to the first pin (plaintext letter a) is routed through 
the three cylinders to appear at the output of the second pin (ciphertext letter B). 
With multiple cylinders, the one closest to the operator input rotates one pin position with each

--- Result 2 ---
the use of multiple cylinders, in which the output pins of  one cylinder are connected to the input 
pins of the next. Figure 2.7 shows a three -cylinder system. The left half of the figure shows a 
position in which the input from the operator to the first pin (plaintext letter a) is routed through 
the three cylinders to appear at the output of the second pin (ciphertext letter B). 
With multiple cylinders, the one closest to the operator input rotates one pin position wi

In [12]:
%pip install langchain

Note: you may need to restart the kernel to use updated packages.


In [13]:
%pip install langchain-community

Note: you may need to restart the kernel to use updated packages.


In [14]:
%pip install langchain-core langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Set key
os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY_HERE"


llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest", 
    temperature=0.3
)

# 2. Pull context from your working vectorstore (leaves as-is)
query = "What does this document explain about multiple cylinders and cipher text?"
matching_docs = vectorstore.similarity_search(query, k=3)
context = "\n\n".join([doc.page_content for doc in matching_docs])

# 4. Prompt
prompt = f"""
You are a helpful assistant. Use the following context from the document to answer the question.
If you don't know the answer based on the context, say you don't know.

Context:
{context}

Question: {query}
Answer:
"""

# 5. Get the response
response = llm.invoke(prompt)
print("\n--- GEMINI ANSWER ---")
print(response.content)


--- GEMINI ANSWER ---
[{'type': 'text', 'text': 'Based on the provided context, the document explains the following about multiple cylinders and ciphertext:\n\n* **Multiple Cylinders:** In a system with multiple cylinders, the output pins of one cylinder are connected to the input pins of the next. Additionally, the cylinder closest to the operator\'s input rotates by one pin position with each input.\n* **Ciphertext:** It explains how plaintext is converted into ciphertext through routing. Specifically, an input from the operator to the first pin (such as the plaintext letter "a") is routed through the cylinders (for example, a three-cylinder system) to emerge at the output as a ciphertext letter (such as the ciphertext letter "B").', 'extras': {'signature': 'EqYUCqMUAQw51sftRkZE268RYJqazYkQ4PDh0W08aBOvvzujQfOuQqQfntRVfiQgVMCpKg4bYi4KZWdtLDa0MBQC4CvQ/XvwwMjnCXAPWm6edH2eYCQ3vm/CaKT5LRT5al5yj4k+SOUONdmcTWE0JQPm10jfxPr+aRwmDFfiYmNRYOBg+ycP2Wi4zZ7bUcLbSWeJ+slNVys+bmG/GUGPvgx8wgWc9I4pz9qo